# 07 · The `dtrack` CLI

**Run first:**
```bash
uv run pytest -v tests/unit/dtrack_left/cli tests/unit/test_packaging.py tests/unit/common/test_parity.py
uv run dtrack --help
```
**Read:** `src/dtrack_left/cli/options.py` (shared decorators) and `cli/__init__.py::main`.

| Convention | Why |
| --- | --- |
| stdout = one JSON line, stderr = UTC logs | scripts and schedulers pipe the result to `jq`, logs never mix in |
| exit 0 ok, 1 command-reported failure or abort, 2 usage error | a scheduler tells "bad data" from "bad command" by the code alone |
| `DTRACK_*` env var behind most options | the same command in a shell, cron or CI, settings in `.env` |

In [ ]:
import json
import os

import nbkit
import pandas as pd
from nbkit import code, show, table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

In [ ]:
import click

from dtrack_left.cli import cli

rows = []
for name, command in sorted(cli.commands.items()):
    for param in command.params:
        if isinstance(param, click.Option):
            rows.append(
                {
                    "command": name,
                    "option": "/".join(param.opts),
                    "env": param.envvar or "",
                    "default": ""
                    if param.required or param.default is None or "UNSET" in str(param.default)
                    else param.default,
                    "required": param.required,
                    "help": param.help,
                }
            )
table(pd.DataFrame(rows).astype({"required": str}))

## Design notes per option

- `--config` (`DTRACK_CONFIG`): loaded and validated before any S3 or database call.
- `--s3` (`DTRACK_S3`): the **run folder**. The decorator builds the store, so commands only see
  `store` and tests can pass a memory store. `fetch-report` reuses it as `--report-s3`.
- `--pair` (repeatable): rerun one pair.
- `--log-level` (`DTRACK_LOG_LEVEL`): eager, so config loading already logs at that level.
- `row-check --workers`: pairs counted in parallel, one database connection each.
- `col-check --pull-workers` (database I/O) and `--stat-workers` (CPU) are separate because the
  bottlenecks differ and pulling overlaps computing.
- `col-check --wip`, `--wait-timeout`, `--poll-interval`: back-pressure and patience (notebook 05).
- `col-check` has no `--config`: the manifest carries everything (notebook 01).

In [ ]:
import subprocess
import sys


def dtrack(*args, **env):
    done = subprocess.run(
        [sys.executable, "-m", "dtrack_left", *args],
        capture_output=True,
        text=True,
        check=False,
        cwd=nbkit.ROOT,
        env={**os.environ, **env},
    )
    print(
        f"$ dtrack {' '.join(args)}   ->   exit {done.returncode}".replace(
            str(nbkit.ROOT), "<repo>"
        )
    )
    if done.stdout.strip():
        show(json.loads(done.stdout.replace(str(nbkit.ROOT), "<repo>")), "stdout")
    if done.stderr.strip():
        code("\n".join(done.stderr.strip().splitlines()[-3:]), label="stderr")


dtrack("doctor", "--config", "examples/e2e/config.json")

`doctor` logs nothing at INFO, so stderr is empty. `row-check` does log: below it runs
in-process against a Moto bucket, with stdout and stderr captured separately.

In [ ]:
import contextlib
import io

import boto3
from moto import mock_aws

from dtrack_left.cli import main

folder = nbkit.sandbox("07")
_, config, _ = nbkit.dataset(folder)
(folder / "config.json").write_text(json.dumps(config))
os.environ |= {
    "AWS_ACCESS_KEY_ID": "x",
    "AWS_SECRET_ACCESS_KEY": "x",
    "AWS_DEFAULT_REGION": "us-east-1",
}
os.environ.pop("DTRACK_S3_ENDPOINT_URL", None)
out, err = io.StringIO(), io.StringIO()
with mock_aws(), contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
    boto3.client("s3").create_bucket(Bucket="demo")
    exit_code = main(
        ["row-check", "--config", str(folder / "config.json"), "--s3", "s3://demo/runs/nb07"]
    )
print("exit:", exit_code)
show(json.loads(out.getvalue()), "stdout")
code("\n".join(err.getvalue().splitlines()[:4]), label="stderr (first lines)")

In [ ]:
broken = json.loads(json.dumps(config))
broken["pairs"]["orders"]["left"]["path"] = str(folder / "missing.csv")
(folder / "broken.json").write_text(json.dumps(broken))
dtrack("doctor", "--config", str(folder / "broken.json"))
dtrack("doctor")

## Run id = the `--s3` folder (a bug found by this course)

Before the fix, `--s3 .../runs/A --run-id B` stored data under `A` while every document and
the report said `B`. Now the run id is the last folder of `--s3`; `--run-id` may repeat it and
must match, otherwise the command stops with a usage error before touching anything.

In [ ]:
dtrack(
    "row-check", "--config", str(folder / "config.json"), "--s3", "s3://b/runs/A", "--run-id", "B"
)
dtrack("row-check", "--config", str(folder / "config.json"), "--s3", "s3://b")